# 第7章: 機械学習

本章では、[Stanford Sentiment Treebank (SST)](https://nlp.stanford.edu/sentiment/) データセットを用い、評判分析器（ポジネガ分類器）を構築する。ここでは処理を簡略化するため、[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されているSSTデータセットを用いる。


## 60. データの入手・整形

GLUEのウェブサイトから[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)データセットを取得せよ。学習データ（`train.tsv`）と検証データ（`dev.tsv`）のぞれぞれについて、ポジティブ (1) とネガティブ (0) の事例数をカウントせよ。

In [1]:
import requests
import zipfile
import io
import os
import pandas as pd

# ダウンロードURL
url = "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip"

# ファイル名
zip_file_name = "SST-2.zip"

# データを保存するディレクトリ
data_dir = "SST-2"

# データディレクトリが存在しない場合は作成
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

print(f"Downloading {zip_file_name}...")
response = requests.get(url)

if response.status_code == 200:
    # zipファイルをメモリ上で展開
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        # 全てのファイルを指定したディレクトリに展開
        zf.extractall(data_dir)
    print(f"{zip_file_name} downloaded and extracted to {data_dir}/")
else:
    print(f"Failed to download the file. Status code: {response.status_code}")

SST-2.zip downloaded and extracted to SST-2/


In [3]:
import os

data_dir = "SST-2"

# SST-2ディレクトリのコンテンツをリスト表示
print(f"Contents of {data_dir}/:")
for root, dirs, files in os.walk(data_dir):
    level = root.replace(data_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Contents of SST-2/:
SST-2/
    SST-2/
        dev.tsv
        test.tsv
        train.tsv
        original/
            sentiment_labels.txt
            datasetSentences.txt
            dictionary.txt
            STree.txt
            datasetSplit.txt
            README.txt
            SOStr.txt
            original_rt_snippets.txt


In [4]:
import pandas as pd
import os

data_dir = os.path.join("SST-2", "SST-2")

# train.tsv の読み込みとカウント
train_path = os.path.join(data_dir, "train.tsv")
train_df = pd.read_csv(train_path, sep='\t')
print("Train data label counts:")
print(train_df['label'].value_counts())

# dev.tsv の読み込みとカウント
dev_path = os.path.join(data_dir, "dev.tsv")
dev_df = pd.read_csv(dev_path, sep='\t')
print("\nDev data label counts:")
print(dev_df['label'].value_counts())

Train data label counts:
label
1    37569
0    29780
Name: count, dtype: int64

Dev data label counts:
label
1    444
0    428
Name: count, dtype: int64


## 61. 特徴ベクトル

Bag of Words (BoW) に基づき、学習データ（`train.tsv`）および検証データ（`dev.tsv`）のテキストを特徴ベクトルに変換したい。ここで、ある事例のテキストの特徴ベクトルは、テキスト中に含まれる単語（スペース区切りのトークン）の出現頻度で構成する。例えば、"too loud , too goofy"というテキストに対応する特徴ベクトルは、以下のような辞書オブジェクトで表現される。

```python
{'too': 2, 'loud': 1, ',': 1, 'goofy': 1}
```

各事例はテキスト、特徴ベクトル、ラベルを格納した辞書オブジェクトでまとめておく。例えば、先ほどの"too loud , too goofy"に対してラベル"0"（ネガティブ）が付与された事例は、以下のオブジェクトで表現される。

```python
{'text': 'too loud , too goofy', 'label': '0', 'feature': {'too': 2, 'loud': 1, ',': 1, 'goofy': 1}}
```

学習データと検証データの各事例を上記のような辞書オブジェクトに変換したうえで、学習データと検証データのそれぞれを、辞書オブジェクトのリストとして表現せよ。さらに、学習データの最初の事例について、正しく特徴ベクトルに変換できたか、目視で確認せよ。

In [8]:
import pandas as pd
import os

data_dir = os.path.join("SST-2", "SST-2")

# train.tsv の処理
train_path = os.path.join(data_dir, "train.tsv")
train_df = pd.read_csv(train_path, sep='\t')

train_features = []
for index, row in train_df.iterrows():
    text = row['sentence']
    label = row['label']
    feature = {}
    for token in text.split():
        if token in feature:
            feature[token] += 1
        else:
            feature[token] = 1
    train_features.append({'text': text, 'label': label, 'feature': feature})

# リストからDataFrameを再構築（または新しい列を追加）
train_processed_df = pd.DataFrame(train_features)

print("Train data (first example with feature vector):")
print(train_processed_df.iloc[0]) # Modified from train_processed_df[0]


# dev.tsv の処理
dev_path = os.path.join(data_dir, "dev.tsv")
dev_df = pd.read_csv(dev_path, sep='\t')

dev_features = []
for index, row in dev_df.iterrows():
    text = row['sentence']
    label = row['label']
    feature = {}
    for token in text.split():
        if token in feature:
            feature[token] += 1
        else:
            feature[token] = 1
    dev_features.append({'text': text, 'label': label, 'feature': feature})

# リストからDataFrameを再構築（または新しい列を追加）
dev_processed_df = pd.DataFrame(dev_features)

print("\nDev data (first example with feature vector):")
print(dev_processed_df.iloc[0])

Train data (first example with feature vector):
text            hide new secretions from the parental units 
label                                                      0
feature    {'hide': 1, 'new': 1, 'secretions': 1, 'from':...
Name: 0, dtype: object

Dev data (first example with feature vector):
text         it 's a charming and often affecting journey . 
label                                                      1
feature    {'it': 1, ''s': 1, 'a': 1, 'charming': 1, 'and...
Name: 0, dtype: object


## 62. 学習

61で構築した学習データの特徴ベクトルを用いて、ロジスティック回帰モデルを学習せよ。

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
import numpy as np

# 特徴ベクトルとラベルを準備
X_train_features = [d['feature'] for d in train_features]
y_train = np.array([d['label'] for d in train_features])

# DictVectorizerを使用して特徴ベクトルを数値データに変換
# 全てのユニークな単語をボキャブラリとして学習
vectorizer = DictVectorizer()
X_train = vectorizer.fit_transform(X_train_features)

# ロジスティック回帰モデルの学習
# solver='liblinear' は小規模データセットに適しており、L1/L2正則化をサポートします。
model = LogisticRegression(solver='liblinear', max_iter = 500)
model.fit(X_train, y_train)

print("ロジスティック回帰モデルの学習が完了しました。")
print(f"学習されたクラス: {model.classes_}")

ロジスティック回帰モデルの学習が完了しました。
学習されたクラス: [0 1]


## 63. 予測

学習したロジスティック回帰モデルを用い、検証データの先頭の事例のラベル（ポジネガ）を予測せよ。また、予測されたラベルが検証データで付与されていたラベルと一致しているか、確認せよ。

In [37]:
from sklearn.feature_extraction import DictVectorizer
import numpy as np

# 検証データの最初の事例を取得
first_dev_example = dev_features[0]
original_text = first_dev_example['text']
original_label = first_dev_example['label']
first_dev_feature = [first_dev_example['feature']]

# DictVectorizerで特徴ベクトルを変換
# 学習データでfitしたvectorizerを使用することが重要です
X_dev_first = vectorizer.transform(first_dev_feature)

# 予測
predicted_label = model.predict(X_dev_first)[0]

print(f"検証データの最初の事例のテキスト: '{original_text}'")
print(f"検証データの最初の事例の元のラベル: {original_label}")
print(f"予測されたラベル: {predicted_label}")

if predicted_label == original_label:
    print("予測されたラベルは元のラベルと一致しています。")
else:
    print("予測されたラベルは元のラベルと一致していません。")

検証データの最初の事例のテキスト: 'it 's a charming and often affecting journey . '
検証データの最初の事例の元のラベル: 1
予測されたラベル: 1
予測されたラベルは元のラベルと一致しています。


## 64. 条件付き確率

学習したロジスティック回帰モデルを用い、検証データの先頭の事例を各ラベル（ポジネガ）に分類するときの条件付き確率を求めよ。

In [38]:
# 検証データの最初の事例を取得（前の問題で定義済み）
# first_dev_example, original_text, original_label, first_dev_feature, X_dev_first は既に定義されています。

# 条件付き確率の取得
# model.predict_proba() は各クラスに属する確率を返します
probabilities = model.predict_proba(X_dev_first)[0]

# クラスラベルと対応する確率を表示
print(f"検証データの最初の事例のテキスト: '{original_text}'")
print(f"検証データの最初の事例の元のラベル: {original_label}")
print("各クラスへの条件付き確率:")
for i, prob in enumerate(probabilities):
    print(f"  クラス {model.classes_[i]}: {prob:.4f}")

検証データの最初の事例のテキスト: 'it 's a charming and often affecting journey . '
検証データの最初の事例の元のラベル: 1
各クラスへの条件付き確率:
  クラス 0: 0.0042
  クラス 1: 0.9958


## 65. テキストのポジネガの予測

与えられたテキストのポジネガを予測するプログラムを実装せよ。例えば、テキストとして"the worst movie I 've ever seen"を与え、ロジスティック回帰モデルの予測結果を確認せよ。


In [39]:
text = "the worst movie I 've ever seen"

def predict_sentiment(text, model, vectorizer):
  probabilities = model.predict_proba(vectorizer.transform([{'text': text}]))[0]
  print(f"テキスト: '{text}'")
  print("各クラスへの条件付き確率:")
  for i, prob in enumerate(probabilities):
    print(f"  クラス {model.classes_[i]}: {prob:.4f}")
  if probabilities[0] > probabilities[1]:
    print(f"予測されたラベル: {model.classes_[0]}")
  else:
    print(f"予測されたラベル: {model.classes_[1]}")

predict_sentiment(text, model, vectorizer)


テキスト: 'the worst movie I 've ever seen'
各クラスへの条件付き確率:
  クラス 0: 0.4146
  クラス 1: 0.5854
予測されたラベル: 1


## 66. 混同行列の作成

学習したロジスティック回帰モデルの検証データにおける混同行列（confusion matrix）を求めよ。

## 67. 精度の計測

学習したロジスティック回帰モデルの正解率、適合率、再現率、F1スコアを、学習データおよび検証データ上で計測せよ。

## 68. 特徴量の重みの確認

学習したロジスティック回帰モデルの中で、重みの高い特徴量トップ20と、重みの低い特徴量トップ20を確認せよ。

## 69. 正則化パラメータの変更

ロジスティック回帰モデルを学習するとき、正則化の係数（ハイパーパラメータ）を調整することで、学習時の適合度合いを制御できる。正則化の係数を変化させながらロジスティック回帰モデルを学習し、検証データ上の正解率を求めよ。実験の結果は、正則化パラメータを横軸、正解率を縦軸としたグラフにまとめよ。